# Hands-On LangChain — Part 3: Document Splitting, Part 2

This maintained companion notebook demonstrates current PDF document splitting, token-based splitting, and Markdown header-aware splitting. It creates its own two-page PDF and Markdown fixture, executes locally, and requires no API key.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-community>=0.3" "langchain-text-splitters>=0.3" pypdf tiktoken reportlab

## 2. Create a reproducible two-page PDF

In [ ]:
from pathlib import Path
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

DATA_DIR = Path("langchain_document_splitting_samples")
DATA_DIR.mkdir(exist_ok=True)
pdf_path = DATA_DIR / "sample.pdf"

pdf = canvas.Canvas(str(pdf_path), pagesize=letter)
pdf.drawString(72, 720, "LangChain loads documents before splitting them into useful chunks.")
pdf.drawString(72, 690, "Chunk overlap helps preserve context between neighboring passages.")
pdf.showPage()
pdf.drawString(72, 720, "Token splitting measures chunk size with the model tokenizer.")
pdf.drawString(72, 690, "Header-aware splitting preserves the section context of Markdown content.")
pdf.save()
print(f"Created {pdf_path} with 2 pages.")

## 3. Load and recursively split the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader(str(pdf_path)).load()
pdf_splitter = RecursiveCharacterTextSplitter(chunk_size=70, chunk_overlap=15)
pdf_chunks = pdf_splitter.split_documents(pages)

print(f"Loaded PDF pages: {len(pages)}")
print(f"Created PDF chunks: {len(pdf_chunks)}")
for index, chunk in enumerate(pdf_chunks, start=1):
    print(f"PDF chunk {index} (page {chunk.metadata['page']}): {chunk.page_content}")

## 4. Split text by model tokens

In [ ]:
from langchain_text_splitters import TokenTextSplitter

token_text = (
    "LangChain applications retrieve compact passages so language models can answer "
    "with focused evidence while retaining the context needed for a useful response."
)
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base", chunk_size=12, chunk_overlap=3
)
token_chunks = token_splitter.split_text(token_text)
print(f"Created token chunks: {len(token_chunks)}")
for index, chunk in enumerate(token_chunks, start=1):
    print(f"Token chunk {index}: {chunk}")

## 5. Preserve Markdown header context

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = '''# Data Platform

The platform stores governed datasets.

## Ingestion

Batch and streaming jobs collect source data.

## Retrieval

Vector search returns passages for grounded answers.
'''
headers_to_split_on = [("#", "Title"), ("##", "Section")]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
markdown_chunks = markdown_splitter.split_text(markdown_text)

print(f"Created Markdown chunks: {len(markdown_chunks)}")
for index, chunk in enumerate(markdown_chunks, start=1):
    print(f"Markdown chunk {index}: metadata={chunk.metadata}; content={chunk.page_content}")

## 6. Verify every result

In [ ]:
assert len(pages) == 2
assert len(pdf_chunks) >= 4
assert {chunk.metadata["page"] for chunk in pdf_chunks} == {0, 1}
assert len(token_chunks) > 1
assert all(chunk.strip() for chunk in token_chunks)
assert len(markdown_chunks) == 3
assert markdown_chunks[1].metadata == {"Title": "Data Platform", "Section": "Ingestion"}
assert markdown_chunks[2].metadata == {"Title": "Data Platform", "Section": "Retrieval"}
print("PDF pages verified: 2")
print(f"PDF chunks verified: {len(pdf_chunks)}")
print(f"Token chunks verified: {len(token_chunks)}")
print(f"Markdown chunks verified: {len(markdown_chunks)}")
print("All Part 2 splitting assertions passed.")